In [1]:
# %aimport helper, tests
# %autoreload 1

In [58]:
import collections

import helper
import numpy as np
import project_tests as tests

import tensorflow as tf
from tensorflow.keras.preprocessing.text import Tokenizer
from tensorflow.keras.preprocessing.sequence import pad_sequences
from tensorflow.keras.models import Model, Sequential
from tensorflow.keras.layers import (
    GRU,
    Input,
    Dense,
    TimeDistributed,
    RepeatVector,
    Bidirectional,
    Embedding,
    Dropout,
)
from tensorflow.keras.callbacks import EarlyStopping, ReduceLROnPlateau

from tensorflow.keras.optimizers import Adam
from tensorflow.keras.losses import sparse_categorical_crossentropy
import pandas as pd
import nltk

from nltk.translate.bleu_score import sentence_bleu, SmoothingFunction

In [ ]:
from tensorflow.keras.callbacks import EarlyStopping, ReduceLROnPlateau

## Dataset

### Load Data dummy tokenized vocab


***REAL CORPUS***

In [3]:
df_real = pd.read_csv('../dataset/REAL_Corpus/gold_translation/df_REAL_benepar_parsed_translated.csv')
df_refer = pd.read_csv('../dataset/refer/referitdataset/gold_translation/df_referit_benepar_parsed_translated.csv')
df_asian_treebank = pd.read_csv('../dataset/AsianTreebank/data/asian_treebank_train.csv')
print('Dataset Loaded')

Dataset Loaded


In [4]:
ast_sentences_id = df_asian_treebank['id']
ast_sentences_en = df_asian_treebank['en']

real_sentences_en = df_real['annotation']
real_sentences_id = df_real['gold_translation']

refer_sentences_en = df_refer['annotation']
refer_sentences_id = df_refer['gold_translation']

In [5]:
english_sentences = helper.load_data('data_dummy/small_vocab_en')
french_sentences = helper.load_data('data_dummy/small_vocab_fr')

### Files


In [6]:
for sample_i in range(5,7):
    print('small_vocab_en Line {}:  {}'.format(sample_i + 1, refer_sentences_en[sample_i]))
    print('small_vocab_fr Line {}:  {}'.format(sample_i + 1, refer_sentences_id[sample_i]))

small_vocab_en Line 6:  water to left of object in center screen
small_vocab_fr Line 6:  air ke kiri objek di layar tengah
small_vocab_en Line 7:  fourth flag from left
small_vocab_fr Line 7:  Bendera keempat dari kiri



### Vocabulary


In [7]:
english_words_counter = collections.Counter([word for sentence in refer_sentences_en for word in sentence.split()])
id_words_counter = collections.Counter([word for sentence in refer_sentences_id for word in sentence.split()])

print('{} English words.'.format(len([word for sentence in refer_sentences_en for word in sentence.split()])))
print('{} unique English words.'.format(len(english_words_counter)))
print('10 Most common words in the English dataset:')
print('"' + '" "'.join(list(zip(*english_words_counter.most_common(10)))[0]) + '"')
print()
print('{} Indonesian words.'.format(len([word for sentence in refer_sentences_id for word in sentence.split()])))
print('{} unique Indonesian words.'.format(len(id_words_counter)))
print('10 Most common words in the Indonesian dataset:')
print('"' + '" "'.join(list(zip(*id_words_counter.most_common(10)))[0]) + '"')

7068 English words.
862 unique English words.
10 Most common words in the English dataset:
"the" "right" "of" "on" "left" "in" "to" "bottom" "top" "middle"

6498 Indonesian words.
1054 unique Indonesian words.
10 Most common words in the Indonesian dataset:
"di" "kanan" "kiri" "sebelah" "atas" "bawah" "tengah" "orang" "dengan" "dari"


For comparison, _Alice's Adventures in Wonderland_ contains 2,766 unique words of a total of 15,500 words.
## Preprocess


Time to start preprocessing the data...
### Tokenize

In [8]:
def tokenize(x):
    """
    Tokenize x
    :param x: List of sentences/strings to be tokenized
    :return: Tuple of (tokenized x data, tokenizer used to tokenize x)
    """
    x_tk = Tokenizer()
    x_tk.fit_on_texts(x)
    return x_tk.texts_to_sequences(x), x_tk
tests.test_tokenize(tokenize)

# Tokenize Example output
text_tokenized, text_tokenizer = tokenize(refer_sentences_en)
print(text_tokenizer.word_index)
print()
for sample_i, (sent, token_sent) in enumerate(zip(refer_sentences_en[:3], text_tokenized)):
    print('Sequence {} in x'.format(sample_i + 1))
    print('  Input:  {}'.format(sent))
    print('  Output: {}'.format(token_sent))

{'the': 1, 'right': 2, 'of': 3, 'on': 4, 'left': 5, 'in': 6, 'to': 7, 'bottom': 8, 'top': 9, 'middle': 10, 'above': 11, 'white': 12, 'people': 13, 'tree': 14, 'guy': 15, 'sky': 16, 'front': 17, 'man': 18, 'far': 19, 'green': 20, 'center': 21, 'building': 22, 'water': 23, 'person': 24, 'from': 25, 'red': 26, 'at': 27, 'blue': 28, 'with': 29, 'shirt': 30, 'side': 31, 'corner': 32, 'back': 33, 'wall': 34, 'any': 35, 'between': 36, 'grass': 37, 'trees': 38, 'part': 39, 'area': 40, 'ground': 41, 'thing': 42, 'hat': 43, 'and': 44, 'woman': 45, "'s": 46, 'below': 47, 'yellow': 48, 'behind': 49, 'window': 50, 'mountain': 51, 'is': 52, 'picture': 53, 'head': 54, 'not': 55, 'car': 56, 'rocks': 57, 'very': 58, 'sand': 59, 'pic': 60, 'two': 61, 'patch': 62, 'group': 63, 'upper': 64, 'dark': 65, 'under': 66, 'girl': 67, 'rock': 68, 'small': 69, 'just': 70, 'brown': 71, 'a': 72, 'lower': 73, 'second': 74, 'that': 75, 'near': 76, 'standing': 77, 'black': 78, 'hand': 79, 'bush': 80, 'most': 81, 'door'

### Padding

Make sure all the English sequences have the same length and all the Indonesian sequences have the same length by adding padding to the **end** of each sequence using Keras's [`pad_sequences`](https://keras.io/preprocessing/sequence/#pad_sequences) function.

In [9]:
def pad(x, length=None):
    """
    Pad x
    :param x: List of sequences.
    :param length: Length to pad the sequence to.  If None, use length of longest sequence in x.
    :return: Padded numpy array of sequences
    """
    # TODO: Implement
    if length == None:
        length = max([len(sentence) for sentence in x])
    return pad_sequences(x,maxlen=length,padding='post')
tests.test_pad(pad)

# Pad Tokenized output
test_pad = pad(text_tokenized)
for sample_i, (token_sent, pad_sent) in enumerate(zip(text_tokenized[:3], test_pad)):
    print('Sequence {} in x'.format(sample_i + 1))
    print('  Input:  {}'.format(np.array(token_sent)))
    print('  Output: {}'.format(pad_sent))

Sequence 1 in x
  Input:  [74 24 25  2]
  Output: [74 24 25  2  0  0  0  0  0  0  0  0  0  0  0  0  0  0  0  0  0]
Sequence 2 in x
  Input:  [35 39  3 16]
  Output: [35 39  3 16  0  0  0  0  0  0  0  0  0  0  0  0  0  0  0  0  0]
Sequence 3 in x
  Input:  [122   4   1   2]
  Output: [122   4   1   2   0   0   0   0   0   0   0   0   0   0   0   0   0   0
   0   0   0]


### Preprocess Pipeline

In [10]:
def preprocess(x, y):
   
    preprocess_x, x_tk = tokenize(x)
    preprocess_y, y_tk = tokenize(y)

    preprocess_x = pad(preprocess_x)
    preprocess_y = pad(preprocess_y)

    # Keras's sparse_categorical_crossentropy function requires the labels to be in 3 dimensions
    preprocess_y = preprocess_y.reshape(*preprocess_y.shape, 1)

    return preprocess_x, preprocess_y, x_tk, y_tk

preproc_refer_en, preproc_refer_id, en_tokenizer, id_tokenizer =\
    preprocess(refer_sentences_en, refer_sentences_id)
    
max_english_sequence_length = preproc_refer_en.shape[1]
max_indo_sequence_length = preproc_refer_id.shape[1]
english_vocab_size = len(en_tokenizer.word_index)
indonesian_vocab_size = len(id_tokenizer.word_index)

print('Data Preprocessed')
print("Max English sentence length:", max_english_sequence_length)
print("Max indonesian sentence length:", max_indo_sequence_length)
print("English vocabulary size:", english_vocab_size)
print("Indonesian vocabulary size:", indonesian_vocab_size)

Data Preprocessed
Max English sentence length: 21
Max indonesian sentence length: 16
English vocabulary size: 836
Indonesian vocabulary size: 774


## Models

- Model 1 is a simple RNN
- Model 2 is a RNN with Embedding
- Model 3 is a Bidirectional RNN
- Model 4 is an optional Encoder-Decoder RNN


### Ids Back to Text


In [11]:
def logits_to_text(logits, tokenizer):
   
    index_to_words = {id: word for word, id in tokenizer.word_index.items()}
    index_to_words[0] = '<PAD>'

    return ' '.join([index_to_words[prediction] for prediction in np.argmax(logits, 1)])

print('`logits_to_text` function loaded.')

`logits_to_text` function loaded.


### Model 3: Bidirectional RNNs (IMPLEMENTATION)

In [12]:
def bd_model(input_shape, output_sequence_length, english_vocab_size, indonesian_vocab_size):
    """
    Build and train a bidirectional RNN model on x and y
    :param input_shape: Tuple of input shape
    :param output_sequence_length: Length of output sequence
    :param english_vocab_size: Number of unique English words in the dataset
    :param indonesian_vocab_size: Number of unique Indonesian words in the dataset
    :return: Keras model built, but not trained
    """
    #Config Hyperparameters
    learning_rate = 0.01
    
    #Create Model
    inputs = Input(shape=input_shape[1:])
    hidden_layer = Bidirectional(GRU(output_sequence_length, return_sequences=True))(inputs)
    outputs = TimeDistributed(Dense(indonesian_vocab_size, activation='softmax'))(hidden_layer)
    
    #Create Model from parameters defined above
    model = Model(inputs=inputs, outputs=outputs)
    model.compile(loss=sparse_categorical_crossentropy,
                  optimizer=Adam(learning_rate),
                  metrics=['accuracy'])
    
    return model
    
tests.test_bd_model(bd_model)
tmp_x = pad(preproc_refer_en, max_indo_sequence_length)
tmp_x = tmp_x.reshape((-1, preproc_refer_id.shape[-2], 1))

bd_mod = bd_model(
        tmp_x.shape,
    max_indo_sequence_length,
    english_vocab_size + 1,
    indonesian_vocab_size + 1)
bd_mod.summary()
bd_mod.fit(tmp_x, preproc_refer_id, batch_size=1024, epochs=10, validation_split=0.2)
print(logits_to_text(bd_mod.predict(tmp_x[:1])[0], id_tokenizer))


2025-06-18 11:26:46.455425: I metal_plugin/src/device/metal_device.cc:1154] Metal device set to: Apple M1
2025-06-18 11:26:46.455466: I metal_plugin/src/device/metal_device.cc:296] systemMemory: 16.00 GB
2025-06-18 11:26:46.455474: I metal_plugin/src/device/metal_device.cc:313] maxCacheSize: 5.33 GB
2025-06-18 11:26:46.455502: I tensorflow/core/common_runtime/pluggable_device/pluggable_device_factory.cc:305] Could not identify NUMA node of platform GPU ID 0, defaulting to 0. Your kernel may not have been built with NUMA support.
2025-06-18 11:26:46.455526: I tensorflow/core/common_runtime/pluggable_device/pluggable_device_factory.cc:271] Created TensorFlow device (/job:localhost/replica:0/task:0/device:GPU:0 with 0 MB memory) -> physical PluggableDevice (device: 0, name: METAL, pci bus id: <undefined>)


Model: "functional_1"

┏━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━┓
┃ Layer (type)                    ┃ Output Shape           ┃       Param # ┃
┡━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━┩
│ input_layer_1 (InputLayer)      │ (None, 16, 1)          │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ bidirectional_1 (Bidirectional) │ (None, 16, 32)         │         1,824 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ time_distributed_1              │ (None, 16, 775)        │        25,575 │
│ (TimeDistributed)               │                        │               │
└─────────────────────────────────┴────────────────────────┴───────────────┘

 Total params: 27,399 (107.03 KB)

 Trainable params: 27,399 (107.03 KB)

 Non-trainable params: 0 (0.00 B)

Epoch 1/10


2025-06-18 11:26:48.894747: I tensorflow/core/grappler/optimizers/custom_graph_optimizer_registry.cc:117] Plugin optimizer for device_type GPU is enabled.


1/1 ━━━━━━━━━━━━━━━━━━━━ 20s 20s/step - accuracy: 0.4285 - loss: 6.6563 - val_accuracy: 0.6514 - val_loss: 6.6106
Epoch 2/10
1/1 ━━━━━━━━━━━━━━━━━━━━ 5s 5s/step - accuracy: 0.6595 - loss: 6.6100 - val_accuracy: 0.6519 - val_loss: 6.5602
Epoch 3/10
1/1 ━━━━━━━━━━━━━━━━━━━━ 5s 5s/step - accuracy: 0.6603 - loss: 6.5587 - val_accuracy: 0.6519 - val_loss: 6.5018
Epoch 4/10
1/1 ━━━━━━━━━━━━━━━━━━━━ 7s 7s/step - accuracy: 0.6605 - loss: 6.4990 - val_accuracy: 0.6519 - val_loss: 6.4320
Epoch 5/10
1/1 ━━━━━━━━━━━━━━━━━━━━ 6s 6s/step - accuracy: 0.6605 - loss: 6.4277 - val_accuracy: 0.6519 - val_loss: 6.3465
Epoch 6/10
1/1 ━━━━━━━━━━━━━━━━━━━━ 5s 5s/step - accuracy: 0.6605 - loss: 6.3407 - val_accuracy: 0.6524 - val_loss: 6.2392
Epoch 7/10
1/1 ━━━━━━━━━━━━━━━━━━━━ 5s 5s/step - accuracy: 0.6604 - loss: 6.2318 - val_accuracy: 0.6522 - val_loss: 6.1019
Epoch 8/10
1/1 ━━━━━━━━━━━━━━━━━━━━ 5s 5s/step - accuracy: 0.6604 - loss: 6.0930 - val_accuracy: 0.6516 - val_loss: 5.9239
Epoch 9/10
1/1 ━━━━━━━━━━

In [13]:
# 1) Define which examples to print
example_indices = [3,4,5,6,7,9]

# 2) Assuming you have these lists from your preprocessing step:
#    english_sentences = [...]     
#    indonesian_sentences = [...]  

for ex_num, i in enumerate(example_indices, start=1):

    # grab the raw sentences
    input_sentence  = refer_sentences_en[i]
    target_sentence = refer_sentences_id[i]

    # model_input already has the right 3D shape
    model_input = tmp_x[i:i+1]

    # run inference
    pred_logits = bd_mod.predict(model_input)[0]
    predicted_sentence = logits_to_text(pred_logits, id_tokenizer)

    print(f"Example {ex_num}:")
    print(f"  English (Input):     {input_sentence}")
    print(f"  Indonesian (Target): {target_sentence}")
    print(f"  Indonesian (Pred):   {predicted_sentence}")
    print("-" * 60)


1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 77ms/step
Example 1:
  English (Input):     bottom most glass front center
  Indonesian (Target): Pusat depan paling bawah kaca depan
  Indonesian (Pred):   <PAD> <PAD> <PAD> <PAD> <PAD> <PAD> <PAD> <PAD> <PAD> <PAD> <PAD> <PAD> <PAD> <PAD> <PAD> <PAD>
------------------------------------------------------------
1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 73ms/step
Example 2:
  English (Input):     any of the two people
  Indonesian (Target): salah satu dari dua orang
  Indonesian (Pred):   <PAD> <PAD> <PAD> <PAD> <PAD> <PAD> <PAD> <PAD> <PAD> <PAD> <PAD> <PAD> <PAD> <PAD> <PAD> <PAD>
------------------------------------------------------------
1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 74ms/step
Example 3:
  English (Input):     water to left of object in center screen
  Indonesian (Target): air ke kiri objek di layar tengah
  Indonesian (Pred):   yang yang yang <PAD> <PAD> <PAD> <PAD> <PAD> <PAD> <PAD> <PAD> <PAD> <PAD> <PAD> <PAD> <PAD>
-------------------------------------------------

### Model 4: Encoder-Decoder 

In [12]:
def encdec_model(input_shape, output_sequence_length, english_vocab_size, indonesian_vocab_size):
    """
    Build and train an encoder-decoder model on x and y
    :param input_shape: Tuple of input shape
    :param output_sequence_length: Length of output sequence
    :param english_vocab_size: Number of unique English words in the dataset
    :param indonesian_vocab_size: Number of unique Indonesian words in the dataset
    :return: Keras model built, but not trained
    """
    # OPTIONAL: Implement
    learning_rate = 1e-2
    latent_dim = 128
    
    # Encoder
    encoder_input = Input(shape=input_shape[1:])
    encoder_gru = GRU(output_sequence_length)(encoder_input)
    encode_output = Dense(latent_dim,activation='relu')(encoder_gru)
    
    # Config Decode
    decoder_input = RepeatVector(output_sequence_length)(encode_output)
    decoder_gru = GRU(latent_dim,return_sequences=True)(decoder_input)
    output_layer = TimeDistributed(Dense(indonesian_vocab_size,activation='softmax'))
    outputs = output_layer(decoder_gru)
    model = Model(inputs=encoder_input, outputs=outputs)
    model.compile(loss=sparse_categorical_crossentropy,
                  optimizer=Adam(learning_rate),
                  metrics=['accuracy'])
    return model

tests.test_encdec_model(encdec_model)
tmp_x = pad(preproc_refer_en, max_indo_sequence_length)
tmp_x = tmp_x.reshape((-1, preproc_refer_id.shape[-2], 1))

ed_mod = encdec_model(
        tmp_x.shape,
    max_indo_sequence_length,
    english_vocab_size + 1,
    indonesian_vocab_size + 1)
ed_mod.summary()
ed_mod.fit(tmp_x, preproc_refer_id, batch_size=1024, epochs=10, validation_split=0.2)

2025-06-18 12:15:27.411069: I metal_plugin/src/device/metal_device.cc:1154] Metal device set to: Apple M1
2025-06-18 12:15:27.411277: I metal_plugin/src/device/metal_device.cc:296] systemMemory: 16.00 GB
2025-06-18 12:15:27.411289: I metal_plugin/src/device/metal_device.cc:313] maxCacheSize: 5.33 GB
2025-06-18 12:15:27.411320: I tensorflow/core/common_runtime/pluggable_device/pluggable_device_factory.cc:305] Could not identify NUMA node of platform GPU ID 0, defaulting to 0. Your kernel may not have been built with NUMA support.
2025-06-18 12:15:27.411345: I tensorflow/core/common_runtime/pluggable_device/pluggable_device_factory.cc:271] Created TensorFlow device (/job:localhost/replica:0/task:0/device:GPU:0 with 0 MB memory) -> physical PluggableDevice (device: 0, name: METAL, pci bus id: <undefined>)


Model: "functional_1"

┏━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━┓
┃ Layer (type)                    ┃ Output Shape           ┃       Param # ┃
┡━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━┩
│ input_layer_1 (InputLayer)      │ (None, 16, 1)          │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ gru_2 (GRU)                     │ (None, 16)             │           912 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense_2 (Dense)                 │ (None, 128)            │         2,176 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ repeat_vector_1 (RepeatVector)  │ (None, 16, 128)        │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ gru_3 (GRU)                     │ (None, 16, 128)        │        99,072 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ time_distributed_1              │ (None, 16, 775)        │        99,975 │
│ (TimeDistributed)               │                        │               │
└─────────────────────────────────┴────────────────────────┴───────────────┘

 Total params: 202,135 (789.59 KB)

 Trainable params: 202,135 (789.59 KB)

 Non-trainable params: 0 (0.00 B)

Epoch 1/10


2025-06-18 12:15:30.906523: I tensorflow/core/grappler/optimizers/custom_graph_optimizer_registry.cc:117] Plugin optimizer for device_type GPU is enabled.


1/1 ━━━━━━━━━━━━━━━━━━━━ 10s 10s/step - accuracy: 0.4287 - loss: 6.6529 - val_accuracy: 0.6511 - val_loss: 6.5993
Epoch 2/10
1/1 ━━━━━━━━━━━━━━━━━━━━ 1s 1s/step - accuracy: 0.6595 - loss: 6.5983 - val_accuracy: 0.6511 - val_loss: 6.2981
Epoch 3/10
1/1 ━━━━━━━━━━━━━━━━━━━━ 1s 707ms/step - accuracy: 0.6595 - loss: 6.2932 - val_accuracy: 0.6511 - val_loss: 4.7531
Epoch 4/10
1/1 ━━━━━━━━━━━━━━━━━━━━ 1s 779ms/step - accuracy: 0.6595 - loss: 4.7319 - val_accuracy: 0.6511 - val_loss: 2.4294
Epoch 5/10
1/1 ━━━━━━━━━━━━━━━━━━━━ 1s 1s/step - accuracy: 0.6595 - loss: 2.3672 - val_accuracy: 0.6511 - val_loss: 2.3358
Epoch 6/10
1/1 ━━━━━━━━━━━━━━━━━━━━ 2s 2s/step - accuracy: 0.6595 - loss: 2.2426 - val_accuracy: 0.6511 - val_loss: 2.5057
Epoch 7/10
1/1 ━━━━━━━━━━━━━━━━━━━━ 1s 799ms/step - accuracy: 0.6595 - loss: 2.3893 - val_accuracy: 0.6511 - val_loss: 2.5661
Epoch 8/10
1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 490ms/step - accuracy: 0.6595 - loss: 2.4308 - val_accuracy: 0.6511 - val_loss: 2.5421
Epoch 9/10
1/

Accuracy keeps increasing from the first epoch to 10nth epoch.

In [ ]:
# 1) Define which examples to print
example_indices = [3,4,5,6,7,9]

# 2) Assuming you have these lists from your preprocessing step:
#    english_sentences = [...]     
#    indonesian_sentences = [...]  

for ex_num, i in enumerate(example_indices, start=1):

    # grab the raw sentences
    input_sentence  = refer_sentences_en[i]
    target_sentence = refer_sentences_id[i]

    # model_input already has the right 3D shape
    model_input = tmp_x[i:i+1]

    # run inference
    pred_logits = ed_mod.predict(model_input)[0]
    predicted_sentence = logits_to_text(pred_logits, id_tokenizer)

    print(f"Example {ex_num}:")
    print(f"  English (Input):     {input_sentence}")
    print(f"  Indonesian (Target): {target_sentence}")
    print(f"  Indonesian (Pred):   {predicted_sentence}")
    print("-" * 60)


1/1 ━━━━━━━━━━━━━━━━━━━━ 2s 2s/step
Example 1:
  English (Input):     bottom most glass front center
  Indonesian (Target): Pusat depan paling bawah kaca depan
  Indonesian (Pred):   <PAD> <PAD> <PAD> <PAD> <PAD> <PAD> <PAD> <PAD> <PAD> <PAD> <PAD> <PAD> <PAD> <PAD> <PAD> <PAD>
------------------------------------------------------------
1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 60ms/step
Example 2:
  English (Input):     any of the two people
  Indonesian (Target): salah satu dari dua orang
  Indonesian (Pred):   <PAD> <PAD> <PAD> <PAD> <PAD> <PAD> <PAD> <PAD> <PAD> <PAD> <PAD> <PAD> <PAD> <PAD> <PAD> <PAD>
------------------------------------------------------------
1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 64ms/step
Example 3:
  English (Input):     water to left of object in center screen
  Indonesian (Target): air ke kiri objek di layar tengah
  Indonesian (Pred):   <PAD> <PAD> <PAD> <PAD> <PAD> <PAD> <PAD> <PAD> <PAD> <PAD> <PAD> <PAD> <PAD> <PAD> <PAD> <PAD>
------------------------------------------------

### Model 5: Custom
incorporates embedding and a bidirectional rnn into one model.

In [44]:
def model_final(input_shape, output_sequence_length, english_vocab_size, indonesian_vocab_size):
    """
    Build and train a model that incorporates embedding, encoder-decoder, and bidirectional RNN on x and y
    :param input_shape: Tuple of input shape
    :param output_sequence_length: Length of output sequeånce
    :param english_vocab_size: Number of unique English words in the dataset
    :param indonesian_vocab_size: Number of unique Indonesian words in the dataset
    :return: Keras model built, but not trained
    """
    #Config Hyperparameters
    learning_rate = 0.01
    latent_dim = 128
    
    #Config Model
    inputs = Input(shape=input_shape[1:])
    embedding_layer = Embedding(input_dim=english_vocab_size,
                                output_dim=output_sequence_length,
                                mask_zero=False)(inputs)
    bd_layer = Bidirectional(GRU(output_sequence_length))(embedding_layer)
    encoding_layer = Dense(latent_dim, activation='relu')(bd_layer)
    decoding_layer = RepeatVector(output_sequence_length)(encoding_layer)
    output_layer = Bidirectional(GRU(latent_dim, return_sequences=True))(decoding_layer)
    outputs = TimeDistributed(Dense(indonesian_vocab_size, activation='softmax'))(output_layer)
    
    #Create Model from parameters defined above
    model = Model(inputs=inputs, outputs=outputs)
    model.compile(
    loss='sparse_categorical_crossentropy',
    optimizer=Adam(1e-2),
    # sample_weight_mode='temporal',   # tells Keras you’ll pass per-timestep weights
    metrics=['accuracy']
)
    
    return model
tests.test_model_final(model_final)
print('Final Model Loaded')



Final Model Loaded


In [13]:
tmp_x = pad(preproc_refer_en, max_indo_sequence_length)
# tmp_x = tmp_x.reshape((-1, preproc_refer_id.shape[-2], 1))

model_bidirect_endec_emb = model_final(
       tmp_x.shape,
    max_indo_sequence_length,
    english_vocab_size + 1,
    indonesian_vocab_size + 1)
model_bidirect_endec_emb.summary()
mask = (preproc_refer_id[...,0] != 0).astype('float32')  

model_bidirect_endec_emb.fit(
  x=tmp_x, 
  y=preproc_refer_id, 
  sample_weight=mask,     # shape (N, seq_len)
  batch_size=1024, 
  epochs=10, 
  validation_split=0.2
)

Model: "functional_1"

┏━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━┓
┃ Layer (type)                    ┃ Output Shape           ┃       Param # ┃
┡━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━┩
│ input_layer_1 (InputLayer)      │ (None, 16)             │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ embedding_1 (Embedding)         │ (None, 16, 16)         │        13,392 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ bidirectional_2 (Bidirectional) │ (None, 32)             │         3,264 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense_2 (Dense)                 │ (None, 128)            │         4,224 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ repeat_vector_1 (RepeatVector)  │ (None, 16, 128)        │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ bidirectional_3 (Bidirectional) │ (None, 16, 256)        │       198,144 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ time_distributed_1              │ (None, 16, 775)        │       199,175 │
│ (TimeDistributed)               │                        │               │
└─────────────────────────────────┴────────────────────────┴───────────────┘

 Total params: 418,199 (1.60 MB)

 Trainable params: 418,199 (1.60 MB)

 Non-trainable params: 0 (0.00 B)

Epoch 1/10


2025-06-18 12:42:37.124300: I tensorflow/core/grappler/optimizers/custom_graph_optimizer_registry.cc:117] Plugin optimizer for device_type GPU is enabled.


1/1 ━━━━━━━━━━━━━━━━━━━━ 22s 22s/step - accuracy: 1.3186e-04 - loss: 2.2656 - val_accuracy: 0.0285 - val_loss: 2.3022
Epoch 2/10
1/1 ━━━━━━━━━━━━━━━━━━━━ 10s 10s/step - accuracy: 0.0279 - loss: 2.2473 - val_accuracy: 0.0430 - val_loss: 1.9393
Epoch 3/10
1/1 ━━━━━━━━━━━━━━━━━━━━ 10s 10s/step - accuracy: 0.0449 - loss: 1.8700 - val_accuracy: 0.0430 - val_loss: 1.9269
Epoch 4/10
1/1 ━━━━━━━━━━━━━━━━━━━━ 10s 10s/step - accuracy: 0.0449 - loss: 1.8160 - val_accuracy: 0.0190 - val_loss: 1.9412
Epoch 5/10
1/1 ━━━━━━━━━━━━━━━━━━━━ 10s 10s/step - accuracy: 0.0160 - loss: 1.8070 - val_accuracy: 0.0084 - val_loss: 1.9437
Epoch 6/10
1/1 ━━━━━━━━━━━━━━━━━━━━ 11s 11s/step - accuracy: 0.0078 - loss: 1.7776 - val_accuracy: 0.0087 - val_loss: 1.9275
Epoch 7/10
1/1 ━━━━━━━━━━━━━━━━━━━━ 10s 10s/step - accuracy: 0.0090 - loss: 1.7400 - val_accuracy: 0.0285 - val_loss: 1.9162
Epoch 8/10
1/1 ━━━━━━━━━━━━━━━━━━━━ 11s 11s/step - accuracy: 0.0280 - loss: 1.7105 - val_accuracy: 0.0430 - val_loss: 1.8843
Epoch 9

In [14]:
# 1) Define which examples to print
example_indices = [3,4,5,6,7,9]

# 2) Assuming you have these lists from your preprocessing step:
#    english_sentences = [...]     
#    indonesian_sentences = [...]  

for ex_num, i in enumerate(example_indices, start=1):

    # grab the raw sentences
    input_sentence  = refer_sentences_en[i]
    target_sentence = refer_sentences_id[i]

    # model_input already has the right 3D shape
    model_input = tmp_x[i:i+1]

    # run inference
    pred_logits = model_bidirect_endec_emb.predict(model_input)[0]
    predicted_sentence = logits_to_text(pred_logits, id_tokenizer)

    print(f"Example {ex_num}:")
    print(f"  English (Input):     {input_sentence}")
    print(f"  Indonesian (Target): {target_sentence}")
    print(f"  Indonesian (Pred):   {predicted_sentence}")
    print("-" * 60)


1/1 ━━━━━━━━━━━━━━━━━━━━ 1s 1s/step
Example 1:
  English (Input):     bottom most glass front center
  Indonesian (Target): Pusat depan paling bawah kaca depan
  Indonesian (Pred):   di di di di di di di di di di di di di di di di
------------------------------------------------------------
1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 116ms/step
Example 2:
  English (Input):     any of the two people
  Indonesian (Target): salah satu dari dua orang
  Indonesian (Pred):   di di di di di di di di di di di di di di di di
------------------------------------------------------------
1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 111ms/step
Example 3:
  English (Input):     water to left of object in center screen
  Indonesian (Target): air ke kiri objek di layar tengah
  Indonesian (Pred):   di di di di di di di di di di di di di di di di
------------------------------------------------------------
1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 102ms/step
Example 4:
  English (Input):     fourth flag from left
  Indonesian (Target): Bendera kee

### Model fix aminnn

In [45]:
from tensorflow.keras.callbacks import EarlyStopping, ReduceLROnPlateau


def model_final_fix(input_shape, output_sequence_length, english_vocab_size, indonesian_vocab_size):
    """
    Fixed version of the Seq2Seq model without attention, using proper embedding dimension, masking,
    bidirectional RNNs, and dropout to reduce overfitting and handle padding correctly.
    :param input_shape: Tuple representing the shape of the input array (including batch dimension)
    :param output_sequence_length: Number of time‐steps in the output sequence
    :param english_vocab_size: Size of the English vocabulary (including the padding token)
    :param indonesian_vocab_size: Size of the Indonesian vocabulary (including the padding token)
    :return: A compiled Keras Model ready for training
    """
    # Config Hyperparameters
    embedding_dim = 128
    latent_dim = 256
    learning_rate = 0.001

    # Build Model
    inputs = Input(shape=input_shape[1:])
    embedding_layer = Embedding(
        input_dim=english_vocab_size,
        output_dim=embedding_dim,
        mask_zero=True
    )(inputs)

    # Encoder: bidirectional GRU with dropout
    encoder = Bidirectional(
        GRU(latent_dim // 2, dropout=0.3, recurrent_dropout=0.2)
    )(embedding_layer)
    encoder = Dense(latent_dim, activation='tanh')(encoder)
    encoder = Dropout(0.3)(encoder)

    # Prepare decoder input
    decoder_input = RepeatVector(output_sequence_length)(encoder)

    # Decoder: bidirectional GRU returning sequences
    decoder = Bidirectional(
        GRU(latent_dim // 2, return_sequences=True, dropout=0.3, recurrent_dropout=0.2)
    )(decoder_input)

    # Final time‐distributed dense layer with softmax over the Indonesian vocab
    outputs = TimeDistributed(
        Dense(indonesian_vocab_size, activation='softmax')
    )(decoder)

    model = Model(inputs=inputs, outputs=outputs)
    model.compile(
        loss='sparse_categorical_crossentropy',
        optimizer=Adam(learning_rate, clipnorm=1.0),
        metrics=['accuracy']
    )
    return model

# Test that the signature is correct
tests.test_model_final(model_final)
print('Final Model Loaded')

# Prepare data
tmp_x = pad(preproc_refer_en, max_indo_sequence_length)

# Instantiate and inspect the model
model_bidirect_endec_emb = model_final(
    tmp_x.shape,
    max_indo_sequence_length,
    english_vocab_size + 1,
    indonesian_vocab_size + 1
)
model_bidirect_endec_emb.summary()

# Create mask for padding tokens
mask = (preproc_refer_id[..., 0] != 0).astype('float32')

# Set up callbacks
callbacks = [
    EarlyStopping(
        monitor='val_loss',
        patience=3,
        restore_best_weights=True,
        verbose=1
    ),
    ReduceLROnPlateau(
        monitor='val_loss',
        factor=0.5,
        patience=3,
        min_lr=1e-6,
        verbose=1
    )
]

# Train the model
model_bidirect_endec_emb.fit(
    x=tmp_x,
    y=preproc_refer_id,
    sample_weight=mask,
    batch_size=128,
    epochs=15,
    validation_split=0.2,
    callbacks=callbacks,
    verbose=1
)


Final Model Loaded


Model: "functional_14"

┏━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━┓
┃ Layer (type)                    ┃ Output Shape           ┃       Param # ┃
┡━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━┩
│ input_layer_14 (InputLayer)     │ (None, 16)             │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ embedding_14 (Embedding)        │ (None, 16, 16)         │        13,392 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ bidirectional_28                │ (None, 32)             │         3,264 │
│ (Bidirectional)                 │                        │               │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense_28 (Dense)                │ (None, 128)            │         4,224 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ repeat_vector_14 (RepeatVector) │ (None, 16, 128)        │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ bidirectional_29                │ (None, 16, 256)        │       198,144 │
│ (Bidirectional)                 │                        │               │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ time_distributed_14             │ (None, 16, 775)        │       199,175 │
│ (TimeDistributed)               │                        │               │
└─────────────────────────────────┴────────────────────────┴───────────────┘

 Total params: 418,199 (1.60 MB)

 Trainable params: 418,199 (1.60 MB)

 Non-trainable params: 0 (0.00 B)

Epoch 1/15
8/8 ━━━━━━━━━━━━━━━━━━━━ 22s 2s/step - accuracy: 0.0291 - loss: 2.0626 - val_accuracy: 0.0430 - val_loss: 1.8518 - learning_rate: 0.0100
Epoch 2/15
8/8 ━━━━━━━━━━━━━━━━━━━━ 10s 1s/step - accuracy: 0.0430 - loss: 1.7042 - val_accuracy: 0.0430 - val_loss: 1.8656 - learning_rate: 0.0100
Epoch 3/15
8/8 ━━━━━━━━━━━━━━━━━━━━ 8s 1s/step - accuracy: 0.0452 - loss: 1.6332 - val_accuracy: 0.0456 - val_loss: 1.8316 - learning_rate: 0.0100
Epoch 4/15
8/8 ━━━━━━━━━━━━━━━━━━━━ 7s 887ms/step - accuracy: 0.0460 - loss: 1.6395 - val_accuracy: 0.0504 - val_loss: 1.8811 - learning_rate: 0.0100
Epoch 5/15
8/8 ━━━━━━━━━━━━━━━━━━━━ 8s 975ms/step - accuracy: 0.0472 - loss: 1.5701 - val_accuracy: 0.0509 - val_loss: 1.8894 - learning_rate: 0.0100
Epoch 6/15
8/8 ━━━━━━━━━━━━━━━━━━━━ 0s 801ms/step - accuracy: 0.0491 - loss: 1.5536
Epoch 6: ReduceLROnPlateau reducing learning rate to 0.004999999888241291.
8/8 ━━━━━━━━━━━━━━━━━━━━ 7s 919ms/step - accuracy: 0.0492 - loss: 1.5542 - val_accuracy: 0.0464 - 

<function __main__.model_final(input_shape, output_sequence_length, english_vocab_size, indonesian_vocab_size)>

In [ ]:
# 1) Define which examples to print
example_indices = [3,4,5,6,7,9]

# 2) Assuming you have these lists from your preprocessing step:
#    english_sentences = [...]     
#    indonesian_sentences = [...]  

for ex_num, i in enumerate(example_indices, start=1):

    # grab the raw sentences
    input_sentence  = refer_sentences_en[i]
    target_sentence = refer_sentences_id[i]

    # model_input already has the right 3D shape
    model_input = tmp_x[i:i+1]

    # run inference
    pred_logits = model_bidirect_endec_emb.predict(model_input)[0]
    predicted_sentence = logits_to_text(pred_logits, id_tokenizer)

    print(f"Example {ex_num}:")
    print(f"  English (Input):     {input_sentence}")
    print(f"  Indonesian (Target): {target_sentence}")
    print(f"  Indonesian (Pred):   {predicted_sentence}")
    print("-" * 60)


1/1 ━━━━━━━━━━━━━━━━━━━━ 1s 1s/step
Example 1:
  English (Input):     bottom most glass front center
  Indonesian (Target): Pusat depan paling bawah kaca depan
  Indonesian (Pred):   di di di di kanan kanan kanan kanan kanan kanan kanan kanan kanan kanan kanan kanan
------------------------------------------------------------
1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 173ms/step
Example 2:
  English (Input):     any of the two people
  Indonesian (Target): salah satu dari dua orang
  Indonesian (Pred):   di di di di kanan kanan kanan kanan kanan kanan kanan kanan kanan kanan kanan kanan
------------------------------------------------------------
1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 122ms/step
Example 3:
  English (Input):     water to left of object in center screen
  Indonesian (Target): air ke kiri objek di layar tengah
  Indonesian (Pred):   di di di di di di di di di di di dengan dengan dengan kepala kepala
------------------------------------------------------------
1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 99ms/step

In [49]:
def get_all_predictions(model, input_data, tokenizer):
    """
    :param model: a trained Keras model that outputs logits
    :param input_data: array of shape (N, seq_len) or (N, seq_len, 1)
    :param tokenizer: the target‐language tokenizer
    :return: list of N decoded sentences
    """
    logits = model.predict(input_data)
    return [logits_to_text(log, tokenizer) for log in logits]

# usage
all_preds_refer = get_all_predictions(model_bidirect_endec_emb, tmp_x, id_tokenizer)

38/38 ━━━━━━━━━━━━━━━━━━━━ 8s 145ms/step


In [52]:
df_refer['predicted_translation_bidirectional'] = all_preds_refer

In [53]:
df_refer

,annotation,parsed,gold_translation,predicted_translation_bidirectional
0,second person from right,(NP (NP (JJ second) (NN person)) (PP (IN from)...,orang kedua dari kanan,di di di di kanan kanan kanan kanan kanan kana...
1,any part of sky,(NP (NP (DT any) (NN part)) (PP (IN of) (NP (N...,Bagian Sky,di di di di kanan kanan kanan kanan kanan kana...
2,fence on the right,(NP (NP (NN fence)) (PP (IN on) (NP (DT the) (...,pagar di sebelah kanan,di di di di kanan kanan kanan kanan kanan kana...
3,bottom most glass front center,(NP (NP (NN bottom)) (NP (JJS most) (NN glass)...,Pusat depan paling bawah kaca depan,di di di di kanan kanan kanan kanan kanan kana...
4,any of the two people,(NP (NP (DT any)) (PP (IN of) (NP (DT the) (CD...,salah satu dari dua orang,di di di di kanan kanan kanan kanan kanan kana...
...,...,...,...,...
1180,blue shirt on right,(NP (NP (JJ blue) (NN shirt)) (PP (IN on) (NP ...,kemeja biru di kanan,di di di di kanan kanan kanan kanan kanan kana...
1181,the sky in the top left that 's actually spide...,(NP (NP (NP (DT the) (NN sky)) (PP (IN in) (NP...,Langit di atas meninggalkan bahwa sebenarnya l...,di di di di di di di di di di di di di di kana...
1182,anywhere at the bottom center,(ADVP (RB anywhere) (PP (IN at) (NP (DT the) (...,Di mana saja di pusat bawah,di di di di kanan kanan kanan kanan kanan kana...
1183,pesrson front left canoe,(NP (NP (NN pesrson) (RB front)) (JJ left) (NN...,Sampan Kiri Depan Pearson,di di di di kanan kanan kanan kanan kanan kana...


In [55]:
# save the model
# model_fixed.save('bidirect_emb_final_fixed.tf')

In [54]:
# loaded_model_final_fix = tf.keras.models.load_model('bidirect_emb_final_fixed.tf')

## Prediction

Model: "functional_28"

┏━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━┓
┃ Layer (type)                    ┃ Output Shape           ┃       Param # ┃
┡━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━┩
│ input_layer_31 (InputLayer)     │ (None, 21)             │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ embedding_16 (Embedding)        │ (None, 21, 16)         │        13,392 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ bidirectional_17                │ (None, 32)             │         3,264 │
│ (Bidirectional)                 │                        │               │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense_37 (Dense)                │ (None, 128)            │         4,224 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ repeat_vector_9 (RepeatVector)  │ (None, 16, 128)        │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ bidirectional_18                │ (None, 16, 256)        │       198,144 │
│ (Bidirectional)                 │                        │               │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ time_distributed_28             │ (None, 16, 775)        │       199,175 │
│ (TimeDistributed)               │                        │               │
└─────────────────────────────────┴────────────────────────┴───────────────┘

 Total params: 418,199 (1.60 MB)

 Trainable params: 418,199 (1.60 MB)

 Non-trainable params: 0 (0.00 B)

Epoch 1/10
1/1 ━━━━━━━━━━━━━━━━━━━━ 20s 20s/step - accuracy: 1.9778e-04 - loss: 6.6514 - val_accuracy: 0.0011 - val_loss: 6.7648
Epoch 2/10
1/1 ━━━━━━━━━━━━━━━━━━━━ 11s 11s/step - accuracy: 0.0011 - loss: 6.7641 - val_accuracy: 0.6511 - val_loss: 3.5035
Epoch 3/10
1/1 ━━━━━━━━━━━━━━━━━━━━ 10s 10s/step - accuracy: 0.6595 - loss: 3.4559 - val_accuracy: 0.6511 - val_loss: 2.9055
Epoch 4/10
1/1 ━━━━━━━━━━━━━━━━━━━━ 11s 11s/step - accuracy: 0.6595 - loss: 2.8130 - val_accuracy: 0.6511 - val_loss: 2.5814
Epoch 5/10
1/1 ━━━━━━━━━━━━━━━━━━━━ 12s 12s/step - accuracy: 0.6595 - loss: 2.4627 - val_accuracy: 0.6511 - val_loss: 2.5830
Epoch 6/10
1/1 ━━━━━━━━━━━━━━━━━━━━ 10s 10s/step - accuracy: 0.6595 - loss: 2.4555 - val_accuracy: 0.6519 - val_loss: 2.5039
Epoch 7/10
1/1 ━━━━━━━━━━━━━━━━━━━━ 11s 11s/step - accuracy: 0.6600 - loss: 2.3624 - val_accuracy: 0.6511 - val_loss: 2.2951
Epoch 8/10
1/1 ━━━━━━━━━━━━━━━━━━━━ 11s 11s/step - accuracy: 0.6595 - loss: 2.1352 - val_accuracy: 0.6511 - val_loss: 2.3

achieved accuracy of 65%

In [ ]:
# 1) Define which examples to print
example_indices = [3,4,5,6,7,9]

# 2) Assuming you have these lists from your preprocessing step:
#    english_sentences = [...]     
#    indonesian_sentences = [...]  

for ex_num, i in enumerate(example_indices, start=1):

    # grab the raw sentences
    input_sentence  = refer_sentences_en[i]
    target_sentence = refer_sentences_id[i]

    # model_input already has the right 3D shape
    model_input = tmp_x[i:i+1]

    # run inference
    pred_logits = model.predict(model_input)[0]
    predicted_sentence = logits_to_text(pred_logits, id_tokenizer)

    print(f"Example {ex_num}:")
    print(f"  English (Input):     {input_sentence}")
    print(f"  Indonesian (Target): {target_sentence}")
    print(f"  Indonesian (Pred):   {predicted_sentence}")
    print("-" * 60)


array([[17, 23,  1, ..., 44,  0,  0],
       [ 5, 20, 21, ..., 51,  2, 45],
       [22,  1,  9, ..., 34,  0,  0],
       ...,
       [24,  1, 10, ..., 54,  0,  0],
       [ 5, 84,  1, ...,  0,  0,  0],
       [ 0,  0,  0, ...,  0,  0,  0]], dtype=int32)

## Evaluation

### BLEU

In [ ]:
# example of BLEU score calculation
hypothesis = ['It', 'is', 'a', 'cat', 'at', 'room']
reference = ['It', 'is', 'a', 'cat', 'inside', 'the', 'room']

BLEUscore = nltk.translate.bleu_score.sentence_bleu([reference], hypothesis)
print(BLEUscore)

0.4548019047027907


In [ ]:

smooth = SmoothingFunction().method1

bleu_scores = []
for ref, hyp in zip(df_refer['gold_translation'], df_refer['predicted_translation_bidirectional']):
    reference = ref.split()
    hypothesis = hyp.split()
    score = sentence_bleu(
        [reference],
        hypothesis,
        smoothing_function=smooth,
        # you can also adjust weights for lower-order BLEU, e.g. BLEU-2:
        weights=(0.5, 0.5, 0, 0)
    )
    bleu_scores.append(score)

average_bleu = np.mean(bleu_scores)
print(f"Average (smoothed) sentence-level BLEU for df_refer: {average_bleu:.4f}")


Average (smoothed) sentence-level BLEU for df_refer: 0.0095


In [64]:
df_refer

,annotation,parsed,gold_translation,predicted_translation_bidirectional
0,second person from right,(NP (NP (JJ second) (NN person)) (PP (IN from)...,orang kedua dari kanan,di di di di kanan kanan kanan kanan kanan kana...
1,any part of sky,(NP (NP (DT any) (NN part)) (PP (IN of) (NP (N...,Bagian Sky,di di di di kanan kanan kanan kanan kanan kana...
2,fence on the right,(NP (NP (NN fence)) (PP (IN on) (NP (DT the) (...,pagar di sebelah kanan,di di di di kanan kanan kanan kanan kanan kana...
3,bottom most glass front center,(NP (NP (NN bottom)) (NP (JJS most) (NN glass)...,Pusat depan paling bawah kaca depan,di di di di kanan kanan kanan kanan kanan kana...
4,any of the two people,(NP (NP (DT any)) (PP (IN of) (NP (DT the) (CD...,salah satu dari dua orang,di di di di kanan kanan kanan kanan kanan kana...
...,...,...,...,...
1180,blue shirt on right,(NP (NP (JJ blue) (NN shirt)) (PP (IN on) (NP ...,kemeja biru di kanan,di di di di kanan kanan kanan kanan kanan kana...
1181,the sky in the top left that 's actually spide...,(NP (NP (NP (DT the) (NN sky)) (PP (IN in) (NP...,Langit di atas meninggalkan bahwa sebenarnya l...,di di di di di di di di di di di di di di kana...
1182,anywhere at the bottom center,(ADVP (RB anywhere) (PP (IN at) (NP (DT the) (...,Di mana saja di pusat bawah,di di di di kanan kanan kanan kanan kanan kana...
1183,pesrson front left canoe,(NP (NP (NN pesrson) (RB front)) (JJ left) (NN...,Sampan Kiri Depan Pearson,di di di di kanan kanan kanan kanan kanan kana...
